## D1 - Building a 3-Class Model: Wall, Red, Blue
Author: George Gorospe, george.gorospe@nmaia.net\
Last Update: July 29th, 2026

### About: Your robot recently learned to sense whether its path was free or blocked. That's useful, but it doesn't know *what* is blocking it. Today you'll teach it to tell the difference between a wall, a red block, and a blue block. This will be your first 3-class model. This skill is highly useful later for our decision-maknig tasks.

### Recall: The Final Layer Decides Class Count
### The first time we trained a network, we learned that a model's final layer determines how many things it can tell apart, in our case, the Red/Blue model's final layer had 2 outputs. Today we are going to show you that you can increase the number of classes any way you like, we're going from 2 classes to 3.
### The good news: `build_model()` already handles this automatically. It builds whatever size final layer matches however many class folders it finds in your dataset, so there's no new code to write for the architecture itself.  
### IMPORTANT: You have to have 3 class folders in your new dataset directory.  
### Same workflow, same functions, the model is just built in a way that it can make one more kind of decision.

### Adding a Third Class: Wall
### What's new: one more folder of data -- `wall/`, alongside `red/` and `blue/`.
### What stays exactly the same: everything else. Same `preview_dataset()`, same hyperparameters, same `prepare_dataloaders()` / `build_model()` / `train_model()` sequence you already know from B3a and B4.3.
### Since we're short on time today, we won't recollect red and blue from scratch -- you already have that data from Wednesday. We'll reuse it, and only collect the one truly new class: wall.

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY D1.1: Reuse Your Red/Blue Data</span>
### Collecting data can be time consuming. In this case since we've already collected some of the data we're just going to copy it to a new directory:
### ```wall_red_blue_dataset```

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 1. Import required libraries

import os
import ipywidgets as widgets
from IPython.display import display

from jetcam_lite import TraitletCamera

from robot_utils import get_rvr, close_if_exists
from gamepad_utils import connect_gamepad, start_control_loop, stop_control_loop
from jupyter_utils import register_click_handler
from preview_utils import register_throttled_preview, encode_jpeg
from capture_utils import ensure_directory, save_image, count_images, copy_class_directory
from train_utils import preview_dataset, prepare_dataloaders, build_model, train_model

### STEP 2. Set up the new dataset folder, and copy your existing red and blue images into it. This is a one-time copy -- your original red_blue dataset is untouched.  
```red_blue_dataset/red``` and ```red_blue_dataset/blue``` --> ```wall_red_blue_dataset```

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####

old_data_dir = os.path.join(os.path.expanduser("~"), "Datasets", "red_blue_dataset")
data_dir = os.path.join(os.path.expanduser("~"), "Datasets", "wall_red_blue_dataset")

wall_directory = os.path.join(data_dir, "wall")
red_directory = os.path.join(data_dir, "red")
blue_directory = os.path.join(data_dir, "blue")

ensure_directory(wall_directory)

red_copied = copy_class_directory(os.path.join(old_data_dir, "red"), red_directory)
blue_copied = copy_class_directory(os.path.join(old_data_dir, "blue"), blue_directory)

print(f"Copied {red_copied} red images and {blue_copied} blue images into {data_dir}")
print(f"Wall images collected so far: {count_images(wall_directory)}")

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY D1.2: Collect Wall Data</span>

### Now let's collect the one new class: wall. Same teleoperation pattern as B4 -- drive around and capture images as you go.
### This time, only the **left bumper** captures an image (labeled `wall`). The right bumper isn't used in this activity. The on-screen button below does the same thing as the left bumper, in case you'd rather use that.
### 🎯 Goal: at least 200 wall images, with real variety -- different walls if your room has more than one, different distances, different angles, different lighting.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 3. Set up the camera, robot connection, and the data collection interface.

rvr = get_rvr()

camera = TraitletCamera()
camera.start()

image_widget = widgets.Image(format='jpeg')
# Only a few frames per second are sent to your laptop -- five robots share
# one access point. The images you SAVE are unaffected; see STEP 4.
register_throttled_preview(camera, image_widget, fps=15, max_width=400, quality=50)

wall_count = widgets.IntText(value=count_images(wall_directory), description='wall', layout=widgets.Layout(width='150px'))
wall_button = widgets.Button(description='Capture Wall', button_style='info')

display(image_widget)
display(widgets.HBox([wall_button, wall_count]))

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 4. Define what happens on a capture request, and wire up both the
# left bumper and the on-screen button to trigger it. The right bumper is
# intentionally left alone -- this activity only collects one new class.

def capture_wall_image(*args):
    # Encode from camera.value, not image_widget.value: the widget holds the
    # small preview that was sent to your laptop, while camera.value is the
    # full-size frame still on the robot. These wall images are encoded
    # exactly the same way as the red/blue images copied in from B2.
    save_image(encode_jpeg(camera.value, quality=95), wall_directory, label='wall')
    wall_count.value = count_images(wall_directory)

def check_gamepad_captures(gamepad_state):
    if gamepad_state.left_bumper_pressed:
        gamepad_state.left_bumper_pressed = False
        capture_wall_image()

register_click_handler(wall_button, capture_wall_image)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 5. Connect the gamepad and start driving + collecting.

gamepad, gamepad_state = connect_gamepad()

left_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Left Stick:',
                                         orientation='vertical', readout=True, readout_format='.2f')
right_stick_slider = widgets.FloatSlider(value=0, min=-1, max=1, step=0.01, description='Right Stick:',
                                          orientation='vertical', readout=True, readout_format='.2f')
display(widgets.HBox([left_stick_slider, right_stick_slider]))


def on_update(left_value, right_value):
    left_stick_slider.value = left_value
    right_stick_slider.value = right_value
    check_gamepad_captures(gamepad_state)


start_control_loop(rvr, gamepad_state, on_update=on_update)

### Drive around and collect at least 200 wall images. When you've collected enough, run the cell below to stop.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
# STEP 6. Stop driving once you've collected enough images.
stop_control_loop()

<span style="color: orange; font-size: 55px; font-style: italic;">ACTIVITY D1.3: Train Your 3-Class Model</span>

<font color='red' size='6'>IMPORTANT: The next part requires the AC adaptor (wall power) shutdown robot and move to wall power.</font>

### Same pattern as B3a and B4.3: preview your data, choose your hyperparameters, then train.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
preview_dataset(data_dir)

### Check your counts and sample images above -- 3 classes this time. Make sure `wall` has at least 200 images with good variety before moving on.

In [ ]:
##### ----- FEEL FREE TO CHANGE THESE VALUES ----- #####
model_name = "wall_red_blue_classifier_v1"  # Default name
epochs = 15
learning_rate = 0.001
momentum = 0.9
batch_size = 16  # keep this at 8 or 16 -- see B3a notes on Jetson memory limits

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
train_loader, test_loader, class_names = prepare_dataloaders(
    data_dir,
    batch_size=batch_size,
    test_fraction=0.2
)

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
model, device = build_model(num_classes=len(class_names))

### Next Step: Start training. This will take about 10 minutes and should be done on wall power. Once it's running, you don't need to wait here. Move on to D2 Part 1 (flowchart design) with your team; that part is paper/whiteboard work and doesn't need the robot or this notebook. We'll come back and check on this model once your flowchart is done.

In [ ]:
##### ----- RUN THIS CELL WITHOUT CHANGING IT ----- #####
history, training_record = train_model(
    model,
    train_loader,
    test_loader,
    device,
    class_names,
    data_dir,
    model_name,
    epochs,
    learning_rate,
    momentum,
)

## Nice work -- your robot now has a 3-class model in progress: wall, red, and blue, all from one training run.

## **NEXT**: Head into D2 Part 1 and start designing your flowchart -- no need to wait for training to finish first.

In [ ]:
#### ------> RUN THIS CELL WHEN YOU'RE DONE WITH THIS NOTEBOOK <-----#####
close_if_exists()
print("Robot connection closed. Safe to move on to the next notebook!")